In [2]:
!pip install Sastrawi
!pip install nltk
import pickle
from pickle import UnpicklingError
import re
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
import requests
import json
import nltk

nltk.download('punkt_tab')
nltk.download('punkt')
nltk.download('stopwords')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 6.9 MB/s eta 0:00:00


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [5]:
import pickle
import re
import string
import requests
import json
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from pickle import UnpicklingError

# =====================================================================
# 1. MEMUAT MODEL & VECTORIZER (Pastikan file sudah di-upload di panel kiri)
# =====================================================================
try:
    with open('/content/svm_model.pkl', 'rb') as f:
        svm = pickle.load(f)
except (EOFError, UnpicklingError) as e:
    print(f"Error loading svm model: {e}")
    print("Check if 'svm_model.pkl' exists and is not empty.")
    print("If the file was transferred, ensure it was done in binary mode.")

try:
    with open('/content/tfidf_vectorizer.pkl', 'rb') as f:
        tfidf = pickle.load(f)
except (EOFError, UnpicklingError) as e:
    print(f"Error loading TF-IDF vectorizer: {e}")

# =====================================================================
# 2. SELEKSI & FUNGSI TEXT PREPROCESSING
# =====================================================================
def cleaningText(text):
    text = re.sub(r'@[A-Za-z0-9]+', '', text)
    text = re.sub(r'#[A-Za-z0-9]+', '', text)
    text = re.sub(r"http\S+", '', text)
    text = re.sub(r'[0-9]+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = text.replace('\n', ' ')
    text = text.translate(str.maketrans('', '', string.punctuation))
    # DISESUAIKAN: Mengganti filter kata BNI menjadi kata terkait Smartfren
    text = ' '.join([word for word in text.split() if word.lower() not in ["smartfren", "kuota", "sinyal", "aplikasi"]])
    text = text.strip(' ')
    return text

def casefoldingText(text):
    return text.lower()

def tokenizingText(text):
    return word_tokenize(text)

def filteringText(text):
    listStopwords = set(stopwords.words('indonesian'))
    listStopwords1 = set(stopwords.words('english'))
    listStopwords.update(listStopwords1)
    listStopwords.update(['iya','yaa','gak','nya','na','sih','ku','di','ya','loh','kah','deh'])
    filtered = []
    for txt in text:
        if txt not in listStopwords:
            filtered.append(txt)
    return filtered

def stemmingText(text):
    factory = StemmerFactory()
    stemmer = factory.create_stemmer()
    words = text.split()
    stemmed_words = [stemmer.stem(word) for word in words]
    return ' '.join(stemmed_words)

def toSentence(list_words):
    return ' '.join(word for word in list_words)

def fix_slangwords(text):
    words = text.split()
    fixed_words = []
    for word in words:
        if word.lower() in slangwords:
            fixed_words.append(slangwords[word.lower()])
        else:
            fixed_words.append(word)
    return ' '.join(fixed_words)

# =====================================================================
# 3. MENGUNDUH KAMUS SLANGWORDS DARI GITHUB DOSEN
# =====================================================================
url = 'https://raw.githubusercontent.com/aninanandah/datasetproject/main/slangwords.json'
response = requests.get(url)

if response.status_code == 200:
    try:
        slangwords = json.loads(response.text)
    except json.JSONDecodeError as e:
        print("Error decoding JSON:", e)
        print("Response content:", response.text)
else:
    print("Failed to fetch data from URL. Status code:", response.status_code)

def preprocess_text(text):
    text = cleaningText(text)
    text = casefoldingText(text)
    text = fix_slangwords(text)
    text = tokenizingText(text)
    text = filteringText(text)
    text = toSentence(text)
    return text

# =====================================================================
# 4. FUNGSI UTAMA PREDIKSI SENTIMEN KEBANGGAAN DOSEN
# =====================================================================
def prediksi_sentimen_kalimat_baru(review_baru, tfidf, svm):
    review_baru_cleaned = cleaningText(review_baru)
    review_baru_casefolded = casefoldingText(review_baru_cleaned)
    review_baru_slangfixed = fix_slangwords(review_baru_casefolded)
    review_baru_tokenized = tokenizingText(review_baru_slangfixed)
    review_baru_filtered = filteringText(review_baru_tokenized)
    review_baru_final = toSentence(review_baru_filtered)

    X_review_baru = tfidf.transform([review_baru_final])
    X_review_baru = X_review_baru.toarray()

    prediksi_sentimen = svm.predict(X_review_baru)

    # OPTIMASI: Mendukung deteksi berbasis teks maupun angka biner (1/0)
    val = prediksi_sentimen[0]
    if val == 'positive' or val == 1 or val == '1':
        hasil = "Sentimen review baru adalah POSITIF."
    elif val == 'negative' or val == 0 or val == '0':
        hasil = "Sentimen review baru adalah NEGATIF."
    else:
        hasil = "Sentimen review baru adalah NETRAL."

    return hasil

In [6]:
review_baru = "Sinyal smartfren di tempat saya sering ngelag dan harganya mahal"
prediksi_sentimen_kalimat_baru(review_baru, tfidf, svm)

'Sentimen review baru adalah POSITIF.'

In [7]:
review_baru = " Sangat membantu, mudah"
prediksi_sentimen_kalimat_baru(review_baru, tfidf, svm)

'Sentimen review baru adalah POSITIF.'

In [9]:
review_baru = "sering eror"
prediksi_sentimen_kalimat_baru(review_baru, tfidf, svm)

'Sentimen review baru adalah NEGATIF.'